# CS570 Week 4 Lab: DataFrames, Spark SQL & Data Investigation

**Points: 100** | **Due: See Canvas**

---

### Instructions

1. Run every code cell and keep all outputs visible
2. Write your code in cells marked `# YOUR CODE`
3. Fill in the **Results Sheet** at the bottom with your exact values
4. Export as PDF (File → Print Preview → Save as PDF)
5. Submit the PDF to Canvas

**Grading:** The Results Sheet is your scorecard. Code cells are your evidence.

---

## Part 0: Setup

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

**Task:** Create a SparkSession. You did this in Week 3.

In [2]:
# YOUR CODE: Create a SparkSession called 'spark'
import os, sys

# Point to Java 17 (PySpark 4.x requires it)
os.environ['JAVA_HOME'] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.13.11-hotspot"
os.environ['PATH'] = os.environ['JAVA_HOME'] + r"\bin;" + os.environ['PATH']
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = SparkSession.builder \
    .appName("CS570 Week 4 Lab") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(f"Spark version: {spark.version}")
print(f"Default parallelism: {sc.defaultParallelism}")


Spark version: 4.1.1
Default parallelism: 16


In [34]:
# Dataset: spotify.csv (same as Week 3)
# If you don't have it, download from the Week 3 lab instructions
data_path = "spotify.csv"

---
## Part 1: Loading & Schema Validation (15 pts)

In production, you always need to validate that your data loaded correctly.

### 1.1 Load with inferSchema

In [33]:
start = time.time()

df_infer = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .csv(data_path)

df_infer.count()  # force evaluation

infer_time = time.time() - start
print(f"inferSchema load time: {infer_time:.2f} seconds")

Py4JJavaError: An error occurred while calling o494.csv.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.Con

### 1.2 Examine the schema

**Task:** Print the schema and show the first few rows. Report the number of columns and rows.

In [5]:
# YOUR CODE: Print the schema that Spark inferred
df_infer.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- track_id: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- explicit: boolean (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)
 |-- track_genre: string (nullable = true)



In [6]:
# YOUR CODE: Show the first 5 rows
df_infer.show(5)

+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|_c0|            track_id|             artists|          album_name|          track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|track_genre|
+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|  0|5SuOikwiRyPMVoIQD...|         Gen Hoshino|              Comedy|              Comedy|        73|     230666|   false|       0.676| 0.461|  1|  -6.746|   0|      0.143|      0.0322|         1.01E-6|   0.358|  0.715| 87.917|            

In [7]:
# YOUR CODE: How many columns? How many rows?
num_cols = len(df_infer.columns)
num_rows = df_infer.count()
print(f"Columns: {num_cols}")
print(f"Rows: {num_rows}")

Columns: 21
Rows: 114000


**Task:** Look at the columns. Are there any that don't add analytical value? Drop them.

In [8]:
# YOUR CODE: Drop any columns that don't add value
# _c0 is just a row index — not analytically useful
df_infer = df_infer.drop("_c0")
print(f"Columns after drop: {len(df_infer.columns)}")
print(df_infer.columns)

Columns after drop: 20
['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


### 1.3 Load with explicit schema

Explicit schemas are faster and catch data problems early.

In [9]:
explicit_schema = StructType([
    StructField("_c0", IntegerType(), True),
    StructField("track_id", StringType(), True),
    StructField("artists", StringType(), True),
    StructField("album_name", StringType(), True),
    StructField("track_name", StringType(), True),
    StructField("popularity", IntegerType(), True),
    StructField("duration_ms", IntegerType(), True),
    StructField("explicit", BooleanType(), True),
    StructField("danceability", DoubleType(), True),
    StructField("energy", DoubleType(), True),
    StructField("key", IntegerType(), True),
    StructField("loudness", DoubleType(), True),
    StructField("mode", IntegerType(), True),
    StructField("speechiness", DoubleType(), True),
    StructField("acousticness", DoubleType(), True),
    StructField("instrumentalness", DoubleType(), True),
    StructField("liveness", DoubleType(), True),
    StructField("valence", DoubleType(), True),
    StructField("tempo", DoubleType(), True),
    StructField("time_signature", IntegerType(), True),
    StructField("track_genre", StringType(), True)
])

start = time.time()

df = spark.read \
    .option("header", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .schema(explicit_schema) \
    .csv(data_path)

df.count()  # force evaluation

explicit_time = time.time() - start
print(f"Explicit schema load time: {explicit_time:.2f} seconds")
print(f"Speedup: {infer_time / explicit_time:.2f}x")

Explicit schema load time: 0.46 seconds
Speedup: 15.12x


**Task:** Verify row counts match, then drop the same useless column from df.

In [10]:
# YOUR CODE: Verify both loaded the same number of rows
# Then drop the useless column from df as well
print(f"inferSchema rows: {df_infer.count()}")
print(f"Explicit schema rows: {df.count()}")
print(f"Match: {df_infer.count() == df.count()}")

df = df.drop("_c0")
print(f"\nColumns after drop: {len(df.columns)}")

inferSchema rows: 114000
Explicit schema rows: 114000
Match: True

Columns after drop: 20


**→ Results Sheet: R1, R2, R3, R4**

---
## Part 2: Data Integrity Investigation (30 pts)

How many unique records does this dataset actually contain?

### 2.1 Find the null row

**Task:** Find which columns have null values. Display the row(s) with nulls. Remove them.

In [11]:
# YOUR CODE: Count nulls in each column
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show(truncate=False)

+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+
|track_id|artists|album_name|track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo|time_signature|track_genre|
+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+
|0       |1      |1         |1         |0         |0          |0       |0           |0     |0  |0       |0   |0          |0           |0               |0       |0      |0    |0             |0          |
+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+-------------

In [12]:
# YOUR CODE: Display the row(s) that have null values
df.filter(
    col("track_name").isNull() | col("artists").isNull() | col("track_id").isNull()
).show(truncate=False)

+----------------------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|track_id              |artists|album_name|track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo  |time_signature|track_genre|
+----------------------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|1kR4gIb7nGxHPI3D2ifs59|NULL   |NULL      |NULL      |0         |0          |false   |0.501       |0.583 |7  |-9.46   |0   |0.0605     |0.69        |0.00396         |0.0747  |0.734  |138.391|4             |k-pop      |
+----------------------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+

In [13]:
# YOUR CODE: Remove the null row(s), store as df_clean
# Print row count before and after

df_clean = df.dropna()

print(f"Rows before: {df.count()}")
print(f"Rows after: {df_clean.count()}")

Rows before: 114000
Rows after: 113999


**→ Results Sheet: R5, R6**

### 2.2 Examine the artists column

**Task:** Look closely at the `artists` column. What do you notice about tracks with multiple artists?

In [14]:
# YOUR CODE: Show some rows where it looks like multiple artists are involved
# Hint: look for a pattern in how they're stored
df_clean.select("artists", "track_name").filter(col("artists").contains(";")).show(10, truncate=False)

+------------------------------------+---------------------+
|artists                             |track_name           |
+------------------------------------+---------------------+
|Ingrid Michaelson;ZAYN              |To Begin Again       |
|A Great Big World;Christina Aguilera|Say Something        |
|Jason Mraz;Colbie Caillat           |Lucky                |
|Chord Overstreet;Deepend            |Hold On - Remix      |
|Andrew Foy;Renee Foy                |ily (i love you baby)|
|Andrew Foy;Renee Foy                |At My Worst          |
|Jason Mraz;Colbie Caillat           |Lucky                |
|Boyce Avenue;Bea Miller             |Photograph           |
|Boyce Avenue;Jennel Garcia          |Demons               |
|A Great Big World;Christina Aguilera|Say Something        |
+------------------------------------+---------------------+
only showing top 10 rows


**→ Results Sheet: R7 — How are multiple artists stored in the artists column?**

### 2.3 How many unique tracks?

**Task:** Determine how many unique tracks are in the dataset. But first — what does "unique" mean? You decide.

In [15]:
# YOUR CODE: Count unique tracks
# First, write your definition:
# MY DEFINITION: A unique track is defined by its track_id (the Spotify unique identifier)
#
# Now implement it:
unique_count = df_clean.select("track_id").distinct().count()
print(f"Unique tracks (by track_id): {unique_count}")

Unique tracks (by track_id): 89740


In [16]:
# YOUR CODE: Compare total rows vs unique tracks
total = df_clean.count()
unique = df_clean.select("track_id").distinct().count()
print(f"Total rows: {total}")
print(f"Unique tracks: {unique}")
print(f"Duplicate rows: {total - unique}")

Total rows: 113999
Unique tracks: 89740
Duplicate rows: 24259


In [17]:
# YOUR CODE: Show the distribution — how many tracks appear 1x, 2x, 3x, etc.
track_counts = df_clean.groupBy("track_id").count()
track_counts.groupBy("count").agg(
    count("*").alias("num_tracks")
).orderBy("count").show()

# Max times a single track appears
max_appearances = track_counts.agg(max("count")).collect()[0][0]
print(f"Max times a single track appears: {max_appearances}")

+-----+----------+
|count|num_tracks|
+-----+----------+
|    1|     73099|
|    2|     11712|
|    3|      2984|
|    4|      1372|
|    5|       431|
|    6|       117|
|    7|        22|
|    8|         2|
|    9|         1|
+-----+----------+

Max times a single track appears: 9


**→ Results Sheet: R8, R9, R10**

### 2.4 Investigate the duplicates

**Task:** Pick a track that appears multiple times. Show all its rows. Determine: do the audio features (energy, danceability) differ? Does the genre differ?

In [18]:
# YOUR CODE: Pick a track that appears 4+ times, show all its rows
# Include: track_id, track_name, artists, track_genre, energy, danceability
frequent_track = df_clean.groupBy("track_id").count().filter(col("count") >= 4).first()["track_id"]

df_clean.filter(col("track_id") == frequent_track) \
    .select("track_id", "track_name", "artists", "track_genre", "energy", "danceability") \
    .show(truncate=False)

+----------------------+----------+------------------------------+-----------+------+------------+
|track_id              |track_name|artists                       |track_genre|energy|danceability|
+----------------------+----------+------------------------------+-----------+------+------------+
|0txb2qyDV7sVBLv2KvtbW2|Bohemio   |Andrés Calamaro;Julio Iglesias|alt-rock   |0.461 |0.784       |
|0txb2qyDV7sVBLv2KvtbW2|Bohemio   |Andrés Calamaro;Julio Iglesias|alternative|0.461 |0.784       |
|0txb2qyDV7sVBLv2KvtbW2|Bohemio   |Andrés Calamaro;Julio Iglesias|latin      |0.461 |0.784       |
|0txb2qyDV7sVBLv2KvtbW2|Bohemio   |Andrés Calamaro;Julio Iglesias|rock       |0.461 |0.784       |
+----------------------+----------+------------------------------+-----------+------+------------+



**Task:** Prove whether duplicates have identical audio features or different ones.

In [19]:
# YOUR CODE: How many track_ids have DIFFERENT energy values across duplicates?
# Hint: groupBy track_id, use countDistinct on energy, filter where > 1
# diff_energy = df_clean.groupBy("track_id") \
#     .agg(countDistinct("energy").alias("distinct_energy")) \
#     .filter(col("distinct_energy") > 1) \
#     .count()
# print(f"Track_ids with differing energy across duplicates: {diff_energy}")
'''
The countDistinct function was renamed in PySpark 4.x. Replace countDistinct with count_distinct
'''
diff_energy = df_clean.groupBy("track_id") \
    .agg(count_distinct("energy").alias("distinct_energy")) \
    .filter(col("distinct_energy") > 1) \
    .count()
print(f"Track_ids with differing energy across duplicates: {diff_energy}")

Track_ids with differing energy across duplicates: 0


In [20]:
# YOUR CODE: How many track_ids have DIFFERENT genre values across duplicates?
# diff_genre = df_clean.groupBy("track_id") \
#     .agg(countDistinct("track_genre").alias("distinct_genre")) \
#     .filter(col("distinct_genre") > 1) \
#     .count()
# print(f"Track_ids with differing genre across duplicates: {diff_genre}")

'''
The countDistinct function was renamed in PySpark 4.x. Replace countDistinct with count_distinct
'''
diff_genre = df_clean.groupBy("track_id") \
    .agg(count_distinct("track_genre").alias("distinct_genre")) \
    .filter(col("distinct_genre") > 1) \
    .count()
print(f"Track_ids with differing genre across duplicates: {diff_genre}")

Track_ids with differing genre across duplicates: 16299


**→ Results Sheet: R11, R12, R13**

---
## Part 3: DataFrame Operations & SQL (20 pts)

In [21]:
# Register for SQL
df_clean.createOrReplaceTempView("tracks")

### 3.1 Impact of duplicates on aggregations

**Task:** Create a deduplicated DataFrame based on your definition of unique track. Then calculate average popularity both ways and compare.

In [22]:
# YOUR CODE: Deduplicate based on YOUR definition of unique track
# Hint: dropDuplicates([...]) is easiest

df_dedup = df_clean.dropDuplicates(["track_id"])

print(f"Rows before dedup: {df_clean.count()}")
print(f"Rows after dedup: {df_dedup.count()}")

Rows before dedup: 113999
Rows after dedup: 89740


In [23]:
# YOUR CODE: Calculate average popularity WITH duplicates and WITHOUT
# Report both values rounded to 2 decimals
avg_with_dup = df_clean.agg(round(avg("popularity"), 2)).collect()[0][0]
avg_without_dup = df_dedup.agg(round(avg("popularity"), 2)).collect()[0][0]

print(f"Avg popularity WITH duplicates:    {avg_with_dup}")
print(f"Avg popularity WITHOUT duplicates: {avg_without_dup}")

Avg popularity WITH duplicates:    33.24
Avg popularity WITHOUT duplicates: 33.2


**→ Results Sheet: R14, R15**

### 3.2 Normalize popularity

**Task:** All audio features are 0.0–1.0. But popularity is 0–100. Add a new column `popularity_norm` that scales it to 0.0–1.0.

In [24]:
# YOUR CODE: Add popularity_norm column using withColumn
# Show 5 rows with track_name, popularity, popularity_norm to verify
df_clean = df_clean.withColumn("popularity_norm", col("popularity") / 100.0)
df_clean.select("track_name", "popularity", "popularity_norm").show(5)

+--------------------+----------+---------------+
|          track_name|popularity|popularity_norm|
+--------------------+----------+---------------+
|              Comedy|        73|           0.73|
|    Ghost - Acoustic|        55|           0.55|
|      To Begin Again|        57|           0.57|
|Can't Help Fallin...|        71|           0.71|
|             Hold On|        82|           0.82|
+--------------------+----------+---------------+
only showing top 5 rows


### 3.3 Unique tracks per genre

**Task:** Using Spark SQL, find how many unique tracks each genre has. Show top 5.

In [25]:
# YOUR CODE: SQL query for unique track count per genre
# Remember: count(*) ≠ unique tracks in denormalized data

# Re-register with the updated df_clean (which may have new columns)
df_clean.createOrReplaceTempView("tracks")

spark.sql("""
    SELECT track_genre, COUNT(DISTINCT track_id) AS unique_tracks
    FROM tracks
    GROUP BY track_genre
    ORDER BY unique_tracks DESC
""").show(5, truncate=False)

+-----------+-------------+
|track_genre|unique_tracks|
+-----------+-------------+
|sad        |1000         |
|synth-pop  |1000         |
|spanish    |1000         |
|rock-n-roll|1000         |
|swedish    |1000         |
+-----------+-------------+
only showing top 5 rows


**→ Results Sheet: R16**

---
## Part 4: Catalyst Optimizer (15 pts)

### 4.1 Optimize this query

**Task:** Look at this inefficient query. Write an optimized version.

In [26]:
# Inefficient query:
query_a = df_clean \
    .filter(col("popularity") > 50) \
    .filter(col("energy") > 0.5) \
    .filter(col("popularity") > 70) \
    .select("track_name", "popularity", "energy")

In [27]:
# YOUR CODE: Write an optimized version
# Think: redundant filters, order of operations
# popularity > 50 is redundant when we also have popularity > 70
# Select first to reduce data width, then filter

query_b = df_clean \
    .select("track_name", "popularity", "energy") \
    .filter((col("popularity") > 70) & (col("energy") > 0.5))

### 4.2 Compare execution plans

**Task:** Run explain() on both queries. What do you notice?

In [1]:
# YOUR CODE: Run explain() on both

print("=== Query A (original): ===")
query_a.explain()

print("\n=== Query B (your version): ===")
query_b.explain()

=== Query A (original): ===


NameError: name 'query_a' is not defined

In [29]:
# YOUR CODE: Verify both return the same number of rows
print(f"Query A rows: {query_a.count()}")
print(f"Query B rows: {query_b.count()}")
print(f"Match: {query_a.count() == query_b.count()}")

Query A rows: 3932
Query B rows: 3932
Match: True


**→ Results Sheet: R17, R18**

---
## Part 5: Pipeline Challenge (20 pts)

**Task:** Build a pipeline to answer this question:

> Which individual artists appear in at least 5 different genres AND have an average track popularity above 50?

**Important:** Remember what you discovered about the `artists` column in Part 2. A track like "Artist A;Artist B;Artist C" should count as appearances for A, B, and C separately.

Requirements:
- Start from df_clean
- Parse the artists column properly
- Show: artist, genre_count, avg_popularity (rounded to 2 decimals)
- Order by genre_count descending, then by avg_popularity descending
- Show top 10

*Hint: Look up `split()` and `explode()` functions.*

In [30]:
# YOUR CODE: Build your pipeline
# Step 1: Split artists by ";" and explode into individual rows
df_artists = df_clean.withColumn("artist", explode(split(col("artists"), ";")))

# Step 2: Trim whitespace from artist names
df_artists = df_artists.withColumn("artist", trim(col("artist")))

# # Step 3: For each artist, count distinct genres and calculate avg popularity
# result = df_artists.groupBy("artist").agg(
#     countDistinct("track_genre").alias("genre_count"),
#     round(avg("popularity"), 2).alias("avg_popularity")
# )
'''
The countDistinct function was renamed in PySpark 4.x. Replace countDistinct with count_distinct
'''
# Step 3: For each artist, count distinct genres and calculate avg popularity
result = df_artists.groupBy("artist").agg(
    count_distinct("track_genre").alias("genre_count"),
    round(avg("popularity"), 2).alias("avg_popularity")
)

# Step 4: Filter to artists in at least 5 genres AND avg popularity > 50
result = result.filter((col("genre_count") >= 5) & (col("avg_popularity") > 50))

# Step 5: Order by genre_count desc, then avg_popularity desc
result = result.orderBy(desc("genre_count"), desc("avg_popularity"))

In [31]:
# YOUR CODE: Show the result
result.show(10, truncate=False)

+--------------+-----------+--------------+
|artist        |genre_count|avg_popularity|
+--------------+-----------+--------------+
|Halsey        |13         |71.47         |
|blackbear     |13         |62.56         |
|Akon          |13         |50.48         |
|Khalid        |12         |64.35         |
|Shreya Ghoshal|12         |57.38         |
|French Montana|12         |57.05         |
|Sia           |12         |50.37         |
|Alan Walker   |11         |62.56         |
|Nicki Minaj   |11         |62.12         |
|Kygo          |11         |61.94         |
+--------------+-----------+--------------+
only showing top 10 rows


**→ Results Sheet: R19, R20**

---
## Part 6: Cleanup

In [32]:
spark.stop()
print("Spark stopped")

Spark stopped


---
---

# RESULTS SHEET

**Fill in every value below. This is what gets graded.**

| # | Question | Your Answer | Points |
|---|----------|-------------|--------|
| R1 | inferSchema load time (seconds) | 6.76 | 3 |
| R2 | Explicit schema load time (seconds) | 0.45 | 3 |
| R3 | Speedup ratio (R1 / R2) | 14.88x | 4 |
| R4 | Number of columns (after dropping useless one) | 20 | 5 |
| R5 | Which columns have nulls? | artists, album_name, track_name (1 row with all three null) | 3 |
| R6 | Row count after removing nulls | 113,999 | 3 |
| R7 | How are multiple artists stored? (describe the format) | Multiple artists are separated by semicolons (`;`), e.g. "Ingrid Michaelson;ZAYN" | 4 |
| R8 | Your unique track count | 89,740 | 4 |
| R9 | Your definition of "unique track" (one sentence) | A unique track is defined by its track_id, the Spotify unique identifier | 5 |
| R10 | Max times a single track appears | 9 | 4 |
| R11 | Track_ids with differing energy across duplicates | 0 | 4 |
| R12 | Track_ids with differing genre across duplicates | 16,299 | 4 |
| R13 | Why do duplicates exist? (one sentence) | The same track is cross-listed across multiple genre playlists, so it appears once per genre with identical audio features but a different genre label | 5 |
| R14 | Avg popularity WITH duplicates (rounded to 2) | 33.24 | 5 |
| R15 | Avg popularity WITHOUT duplicates (rounded to 2) | 33.20 | 5 |
| R16 | Genre with most unique tracks + that count | Multiple genres tied at 1,000 unique tracks each (e.g. sad, synth-pop, spanish, rock-n-roll, swedish) | 6 |
| R17 | Filters in Query A's physical plan | Query A has 3 filters combined into one stage: `popularity > 50`, `energy > 0.5`, `popularity > 70` (plus null checks). The redundant `popularity > 50` is NOT eliminated by Catalyst. | 6 |
| R18 | Are the plans identical? (yes/no + why) | No — Query A still contains the redundant `popularity > 50` filter while Query B only has `popularity > 70` and `energy > 0.5`. Catalyst merges separate filter() calls into one stage but does not eliminate logically redundant predicates. | 7 |
| R19 | Artist appearing in most genres (meeting criteria) | Halsey (13 genres) | 10 |
| R20 | That artist's avg popularity (rounded to 2 decimals) | 71.47 | 10 |
| | | **TOTAL** | **100** |